When working with R, it is often important to understand how long different parts of your code take to execute. This becomes especially relevant when working with large datasets, building models, or optimizing pipelines.

R provides simple tools like `system.time()`, but in practice, we often need a more structured way to measure execution time across multiple steps in a workflow.

In this tutorial, we will build a small reusable timing framework in R that allows us to:

- Measure execution time for individual steps
- Track progress in a pipeline
- Keep the code clean and readable

We will demonstrate this using a simple workflow: data generation, preprocessing, and model training.

In [1]:
# timed_pipeline.R

cat("===== R Script Execution Started =====\n")
cat("Start time:", format(Sys.time()), "\n\n")

# ----------- Helper Function to Time Each Block -----------
time_it <- function(expr, label = "") {
  time <- system.time(result <- eval(expr))
  cat(sprintf(">>> '%s' took %.3f seconds\n", label, time["elapsed"]))
  return(result)
}

# ----------- 1. Load or Generate Data -----------
load_data <- function(n = 1000000) {
  set.seed(42)
  data.frame(
    id = 1:n,
    feature1 = rnorm(n),
    feature2 = runif(n),
    target = sample(0:1, n, replace = TRUE)
  )
}

cat("Step 1: Loading data...\n")
df <- time_it(quote(load_data()), "Load Data")

# ----------- 2. Preprocess Data -----------
preprocess_data <- function(data) {
  data$feature1_scaled <- scale(data$feature1)
  data$feature2_scaled <- scale(data$feature2)
  data
}

cat("\nStep 2: Preprocessing data...\n")
df_processed <- time_it(quote(preprocess_data(df)), "Preprocess Data")

# ----------- 3. Train a Simple Model -----------
train_model <- function(data) {
  model <- glm(target ~ feature1_scaled + feature2_scaled, data = data, family = "binomial")
  return(model)
}

cat("\nStep 3: Training model...\n")
model <- time_it(quote(train_model(df_processed)), "Train Model")

# ----------- 4. Make Predictions -----------
predict_model <- function(model, data) {
  probs <- predict(model, newdata = data, type = "response")
  data$predicted_class <- ifelse(probs > 0.5, 1, 0)
  return(data)
}

cat("\nStep 4: Making predictions...\n")
df_predicted <- time_it(quote(predict_model(model, df_processed)), "Predict")

# ----------- 5. Save Results -----------
save_results <- function(data, path = "output.csv") {
  write.csv(data, path, row.names = FALSE)
}

cat("\nStep 5: Saving output to file...\n")
time_it(quote(save_results(df_predicted)), "Save Output")

cat("\n===== R Script Execution Completed =====\n")
cat("End time:", format(Sys.time()), "\n")


===== R Script Execution Started =====
Start time: 2026-05-01 06:42:45 

Step 1: Loading data...
>>> 'Load Data' took 0.046 seconds

Step 2: Preprocessing data...
>>> 'Preprocess Data' took 0.099 seconds

Step 3: Training model...
>>> 'Train Model' took 0.990 seconds

Step 4: Making predictions...
>>> 'Predict' took 0.149 seconds

Step 5: Saving output to file...
>>> 'Save Output' took 2.835 seconds


NULL


===== R Script Execution Completed =====
End time: 2026-05-01 06:42:49 
